04_jobbank_exploration.ipynb

Exploratory analysis of Canada's Job Bank Open Data
(3 months: February, March, April 2026).

Goal of this notebook:
  - Load all 3 monthly CSV files
  - Concatenate into a single DataFrame
  - Filter to Ontario + 5 data-related NOC codes
  - Sanity-check sample size, salary fill rate, and Salary Per distribution

Source : https://open.canada.ca/data/en/dataset/ea639e28-c0fc-48bf-b5dd-b8899bd43072
Licence: Open Government Licence -- Canada

In [15]:
from pathlib import Path
import pandas as pd

# Display configuration -- show ALL columns when we inspect heads()
pd.set_option("display.max_columns", None)

RAW_DATA_DIR = Path("../data/raw/jobbank")

# Sanity check: confirm the folder exists and lists our 3 files
files = sorted(RAW_DATA_DIR.glob("*.csv"))
print(f"Folder: {RAW_DATA_DIR.resolve()}")
print(f"Files found: {len(files)}")
for f in files:
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f" - {f.name} ({size_mb:.1f} MB)")


Folder: C:\Users\rebec\Documents\GitHub\toronto-data-job-market-analysis\data\raw\jobbank
Files found: 3
 - job-bank-open-data-all-job-postings-en-apr2026.csv (45.2 MB)
 - job-bank-open-data-all-job-postings-en-feb2026.csv (37.6 MB)
 - job-bank-open-data-all-job-postings-en-mar2026.csv (42.2 MB)


In [16]:
# Read each monthly CSV with the encoding/delimiter quirks we discovered:
#   - encoding='utf-16' : the file starts with a BOM (0xff 0xfe) and uses
#                         2-byte chars, NOT utf-8 despite the .csv extension
#   - sep='\t'          : fields are tab-separated, NOT comma-separated
#                         (so technically a TSV file)
#   - low_memory=False  : disables pandas' chunked type inference, which
#                         throws DtypeWarning on mixed-type columns

frames = []  # collect each month's DataFrame, then concat at the end

for f in files:
    print(f"Reading {f.name} ...")
    df_month = pd.read_csv(
        f,
        encoding="utf-16",
        sep="\t",
        low_memory=False,
    )
    # Add a column tracking the source month -- crucial for traceability
    df_month["_source_month"] = f.stem.rsplit("-", 1)[-1]  # e.g. "apr2026"
    frames.append(df_month)
    print(f"   {len(df_month):,} rows x {df_month.shape[1]} columns")

# Concatenate the 3 months into a single DataFrame
df_jobbank = pd.concat(frames, ignore_index=True)

print()
print(f"Combined DataFrame: {len(df_jobbank):,} rows x {df_jobbank.shape[1]} columns")
print()
print("Rows per source month:")
print(df_jobbank["_source_month"].value_counts())

Reading job-bank-open-data-all-job-postings-en-apr2026.csv ...
   53,616 rows x 66 columns
Reading job-bank-open-data-all-job-postings-en-feb2026.csv ...
   44,176 rows x 66 columns
Reading job-bank-open-data-all-job-postings-en-mar2026.csv ...
   49,746 rows x 66 columns

Combined DataFrame: 147,538 rows x 66 columns

Rows per source month:
_source_month
apr2026    53616
mar2026    49746
feb2026    44176
Name: count, dtype: int64


# -----------------------------------------------------------------------------
# Step 1: Filter to Ontario only
# -----------------------------------------------------------------------------
# The 'Province/Territory' column uses full names (not abbreviations like 'ON').
# trailing spaces or unexpected casing in gov data.

In [17]:
print("Province/Territory unique values:")
print(df_jobbank["Province/Territory"].value_counts(dropna=False))
print()

Province/Territory unique values:
Province/Territory
Ontario                      44824
Québec                       29601
British Columbia             28050
Alberta                      17416
Saskatchewan                 11205
Nova Scotia                   5227
Manitoba                      3984
New Brunswick                 3349
Newfoundland and Labrador     2118
Prince Edward Island           930
Yukon                          428
Northwest Territories          306
Nunavut                        100
Name: count, dtype: int64



# -----------------------------------------------------------------------------
# Step 2: Filter to the 5 data-related NOC 2021 codes
# -----------------------------------------------------------------------------
# Reference (from Statistics Canada NOC 2021 taxonomy):
#   21221 -- Data scientists
#   21222 -- Information systems specialists (incl. BI Analysts)
#   21223 -- Database analysts and data administrators
#   21211 -- Mathematicians, statisticians and actuaries
#   12102 -- Statistical officers and related research support

In [18]:
DATA_NOC_CODES = {
    "21211": "Data scientists",
    "21221": "Business systems specialists",
    "21222": "Information systems specialists",
    "21223": "Database analysts and data administrators",
}
# matching, otherwise '2122'!= '21221.0' and our filter returns 0 rows
df_jobbank["_noc21_clean"] = df_jobbank["NOC21 Code"].astype(str).str.replace(".0", "", regex=False)

# Apply both filter : Ontario and data-related NOC codes
mask_ontario = df_jobbank["Province/Territory"] == "Ontario"
mask_data = df_jobbank["_noc21_clean"].isin(DATA_NOC_CODES.keys())

df_data_ontario = df_jobbank[mask_ontario & mask_data].copy()

In [19]:
#Sanity check the results
print(f"Total Canada (3 months): {len(df_jobbank):>7,} rows")
print(f"Ontario only:       {mask_ontario.sum():>7,} rows")
print(f"Ontario + data Noc: {len(df_data_ontario):>7,} rows")
print()

print("Break down by NOC code:")
noc_counts = df_data_ontario["_noc21_clean"].value_counts()
for noc, label  in DATA_NOC_CODES.items():
    n = noc_counts.get(noc, 0)
    print(f" - {noc} -- {label}: {n:,}")
print()
print("Break down by source month:")
print(df_data_ontario["_source_month"].value_counts())

Total Canada (3 months): 147,538 rows
Ontario only:        44,824 rows
Ontario + data Noc:   1,113 rows

Break down by NOC code:
 - 21211 -- Data scientists: 215
 - 21221 -- Business systems specialists: 175
 - 21222 -- Information systems specialists: 651
 - 21223 -- Database analysts and data administrators: 72

Break down by source month:
_source_month
mar2026    409
apr2026    372
feb2026    332
Name: count, dtype: int64


In [22]:
#1 - Salary fill rate

n_total = len(df_data_ontario)
n_with_min = df_data_ontario["Salary Minimum"].notna().sum()
n_with_max = df_data_ontario["Salary Maximum"].notna().sum()
n_with_per = df_data_ontario["Salary Per"].notna().sum()
n_with_all = (
    df_data_ontario["Salary Minimum"].notna()
    & df_data_ontario ["Salary Maximum"].notna()
    & df_data_ontario["Salary Per"].notna()
).sum()

print("Salary fill rate (Ontario data jobs, 3 months combined):")
print(f"  Total jobs:                    {n_total:,}")
print(f"  With Salary Minimum:           {n_with_min:,}  ({100*n_with_min/n_total:.1f}%)")
print(f"  With Salary Maximum:           {n_with_max:,}  ({100*n_with_max/n_total:.1f}%)")
print(f"  With Salary Per:               {n_with_per:,}  ({100*n_with_per/n_total:.1f}%)")
print(f"  With ALL three (min+max+per):  {n_with_all:,}  ({100*n_with_all/n_total:.1f}%)")
print()

#2 - Distribution of Salary Per
print("Salary Per distribution (Ontario data job):")
print(df_data_ontario["Salary Per"].value_counts(dropna=False))
print()

#3 - Cross-tab salary Per x NOC Code
print("Salary Per x Noc Code (rows = Salary Per, cols =Noc):")
salary_per_by_noc = pd.crosstab(
    df_data_ontario["Salary Per"],
    df_data_ontario["_noc21_clean"],
    margins=True,
    margins_name='TOTAL',

)
print(salary_per_by_noc)



Salary fill rate (Ontario data jobs, 3 months combined):
  Total jobs:                    1,113
  With Salary Minimum:           1,113  (100.0%)
  With Salary Maximum:           1,113  (100.0%)
  With Salary Per:               1,113  (100.0%)
  With ALL three (min+max+per):  1,113  (100.0%)

Salary Per distribution (Ontario data job):
Salary Per
Hour         840
Year         227
Day           27
Month         18
Bi-weekly      1
Name: count, dtype: int64

Salary Per x Noc Code (rows = Salary Per, cols =Noc):
_noc21_clean  21211  21221  21222  21223  TOTAL
Salary Per                                     
Bi-weekly         0      1      0      0      1
Day               1      0     26      0     27
Hour            176    162    436     66    840
Month             9      0      8      1     18
Year             29     12    181      5    227
TOTAL           215    175    651     72   1113


# -----------------------------------------------------------------------------
# Step 1: Define a multiplier per "Salary Per" value to convert to ANNUAL
# -----------------------------------------------------------------------------
# Standard Canadian full-time work assumptions:
#   - 40 hours/week, 52 weeks/year     -> 2080 hours/year
#   - 5 working days/week, 52 weeks    -> 260 working days/year (paid)
#                                         (we use 260, not 365)
#   - 12 months/year                    -> 12
#   - bi-weekly  = every 2 weeks        -> 26 pay periods/year
#   - week                              -> 52 pay periods/year

In [23]:
ANNUAL_MULTIPLIERS = {
    "Hour":      2080,   # 40 h/week * 52 weeks
    "Day":        260,   # 5 days/week * 52 weeks
    "Week":        52,
    "Bi-weekly":   26,
    "Month":       12,
    "Year":         1,   # already annual
}
# Sanity check: every Salary Per value in our data has a multiplier defined.
unique_per = df_data_ontario["Salary Per"].unique()
missing = [v for v in unique_per if v not in ANNUAL_MULTIPLIERS]
if missing:
    raise ValueError(f"Unknown Salary Per values, no multiplier defined: {missing}")
print(f"All {len(unique_per)} unique Salary Per values are covered")
print()




All 5 unique Salary Per values are covered



# -----------------------------------------------------------------------------
# Step 2: Compute annualized salary columns (vectorized, no loop)
# -----------------------------------------------------------------------------
# return a new Series of mapped values". Much cleaner than apply(lambda)

In [25]:
multipliers = df_data_ontario["Salary Per"].map(ANNUAL_MULTIPLIERS)
df_data_ontario["salary_min_annual"] = df_data_ontario["Salary Minimum"] * multipliers
df_data_ontario["salary_max_annual"] = df_data_ontario["Salary Maximum"] * multipliers
df_data_ontario["salary_avg_annual"] = (
    df_data_ontario["salary_min_annual"] + df_data_ontario["salary_max_annual"]
) / 2
#Sanity check
print("Annualized salary distribution (all 1,113 Ontario data jobs):")
print(df_data_ontario["salary_avg_annual"].describe().round(0))
print()

# Show 5 lowest and 5 highest for visual inspection
print("5 lowest annualized salaries (sanity check for outliers):")
cols_to_show = ["Job Title", "Salary Minimum", "Salary Maximum",
                "Salary Per", "salary_avg_annual", "_noc21_clean"]
print(df_data_ontario.nsmallest(5, "salary_avg_annual")[cols_to_show])
print()

print("5 highest annualized salaries (sanity check for outliers):")
print(df_data_ontario.nlargest(5, "salary_avg_annual")[cols_to_show])

Annualized salary distribution (all 1,113 Ontario data jobs):
count         1113.0
mean       1366850.0
std       13763812.0
min             22.0
25%          72800.0
50%          96897.0
75%         103730.0
max      194656800.0
Name: salary_avg_annual, dtype: float64

5 lowest annualized salaries (sanity check for outliers):
                                  Job Title  Salary Minimum  Salary Maximum  \
66950   information technology (IT) analyst            21.0            24.0   
112890  information technology (IT) analyst            21.0            24.0   
106921  information technology (IT) analyst          2000.0          3500.0   
112715  information technology (IT) analyst          2000.0          3500.0   
40189   information technology (IT) analyst          3200.0          4000.0   

       Salary Per  salary_avg_annual _noc21_clean  
66950        Year               22.5        21222  
112890       Year               22.5        21222  
106921       Year             2750.0    

In [26]:
# Diagnose how many "unit mislabeling" cases we have
# Heuristic boundaries based on plausibility for Ontario data jobs:
#   - Yearly salaries below $20,000 are sub-minimum-wage -> probably mislabeled hourly/monthly
#   - Hourly salaries above $300/h would imply >$600K/year -> probably mislabeled yearly
#   - Hourly salaries above $1000/h are physically impossible -> certainly yearly

# Case 1: Salary Per = "Year" but values look hourly or monthly
suspicious_low = df_data_ontario[
    (df_data_ontario["Salary Per"] == "Year")
    & (df_data_ontario["Salary Minimum"] < 20000)
]
print(f"Suspicious 'Year' with min < $20K (likely mislabeled): {len(suspicious_low)}")
print()

# Case 2: Salary Per = "Hour" but values look yearly
suspicious_high = df_data_ontario[
    (df_data_ontario["Salary Per"] == "Hour")
    & (df_data_ontario["Salary Minimum"] > 300)
]
print(f"Suspicious 'Hour' with min > $300/h (likely mislabeled): {len(suspicious_high)}")
print()

# Within those, distinguish 'plausible if reinterpreted' from 'truly garbage'
# A yearly salary of $20-100 (current 'Year') most likely meant hourly
# A yearly salary of $1,000-5,000 most likely meant monthly
print("Breakdown of 'Year' suspicious by Salary Minimum range:")
ranges = pd.cut(
    suspicious_low["Salary Minimum"],
    bins=[0, 100, 1000, 10000, 20000],
    labels=["<$100 (hourly?)", "$100-$1K", "$1K-$10K (monthly?)", "$10K-$20K"],
)
print(ranges.value_counts())
print()

print("Breakdown of 'Hour' suspicious by Salary Minimum range:")
ranges_high = pd.cut(
    suspicious_high["Salary Minimum"],
    bins=[300, 1000, 10000, 100000, 1000000],
    labels=["$300-$1K", "$1K-$10K", "$10K-$100K (yearly?)", ">$100K"],
)
print(ranges_high.value_counts())

Suspicious 'Year' with min < $20K (likely mislabeled): 5

Suspicious 'Hour' with min > $300/h (likely mislabeled): 9

Breakdown of 'Year' suspicious by Salary Minimum range:
Salary Minimum
$1K-$10K (monthly?)    3
<$100 (hourly?)        2
$100-$1K               0
$10K-$20K              0
Name: count, dtype: int64

Breakdown of 'Hour' suspicious by Salary Minimum range:
Salary Minimum
$10K-$100K (yearly?)    9
$300-$1K                0
$1K-$10K                0
>$100K                  0
Name: count, dtype: int64


# =============================================================================
# Correction + safety filter
# =============================================================================
# Step 1 -- Build a corrected "Salary Per" column based on plausibility rules
# Step 2 -- Recompute annual salaries using the corrected unit
# Step 3 -- Apply a safety filter for any remaining out-of-range outliers
# Step 4 -- Document everything in a clear audit trail

In [27]:
# 1 - Detect and correct unit mislabeling

# Rule A : "Year" with salary min < 100$ -> probably "Hour"
# Rule B: "Year" with Salary Min in [$1K, $10K] -> probably "Month"
# Rule C: "Hour" with Salary Min in [$10K, $100K] -> probably "Year"

df_data_ontario["_salary_per_corrected"] = df_data_ontario["Salary Per"].copy()

# Rule A: "Year" but values look hourly
mask_a = (
    (df_data_ontario["Salary Per"] == "Year")
    & (df_data_ontario["Salary Minimum"] < 100)
)
df_data_ontario.loc[mask_a, "_salary_per_corrected"] = "Hour"

# Rule B: "Year" but values look monthly
mask_b = (
    (df_data_ontario["Salary Per"] == "Year")
    & (df_data_ontario["Salary Minimum"] >= 1000)
    & (df_data_ontario["Salary Minimum"] < 10000)
)
df_data_ontario.loc[mask_b, "_salary_per_corrected"] = "Month"

# Rule C: "Hour" but values look yearly
mask_c = (
    (df_data_ontario["Salary Per"] == "Hour")
    & (df_data_ontario["Salary Minimum"] >= 10000)
    & (df_data_ontario["Salary Minimum"] < 100000)
)
df_data_ontario.loc[mask_c, "_salary_per_corrected"] = "Year"

n_corrected_a = mask_a.sum()
n_corrected_b = mask_b.sum()
n_corrected_c = mask_c.sum()
n_corrected_total = n_corrected_a + n_corrected_b + n_corrected_c

print("Unit mislabeling corrections applied:")
print(f"  Rule A ('Year' -> 'Hour'):  {n_corrected_a} jobs")
print(f"  Rule B ('Year' -> 'Month'): {n_corrected_b} jobs")
print(f"  Rule C ('Hour' -> 'Year'):  {n_corrected_c} jobs")
print(f"  Total corrected:            {n_corrected_total} jobs ({100*n_corrected_total/len(df_data_ontario):.1f}%)")
print()


Unit mislabeling corrections applied:
  Rule A ('Year' -> 'Hour'):  2 jobs
  Rule B ('Year' -> 'Month'): 3 jobs
  Rule C ('Hour' -> 'Year'):  9 jobs
  Total corrected:            14 jobs (1.3%)



In [28]:
 # 2- Recompute annualized salaries using the CORRECTED unit

multipliers_corrected = df_data_ontario["_salary_per_corrected"].map(ANNUAL_MULTIPLIERS)

df_data_ontario["salary_min_annual"] = (
    df_data_ontario["Salary Minimum"] * multipliers_corrected
)
df_data_ontario["salary_max_annual"] = (
    df_data_ontario["Salary Maximum"] * multipliers_corrected
)
df_data_ontario["salary_avg_annual"] = (
    df_data_ontario["salary_min_annual"] + df_data_ontario["salary_max_annual"]
) / 2

In [29]:
# 3 - Safety filter -- any annualized salary still outside plausible range
#   Lower bound: $25,000  -- below Ontario's annualized minimum wage
#   Upper bound: $500,000 -- above 99th percentile for data roles in Canada


LOWER_BOUND = 25_000
UPPER_BOUND = 500_000

mask_in_range = (
    (df_data_ontario["salary_avg_annual"] >= LOWER_BOUND)
    & (df_data_ontario["salary_avg_annual"] <= UPPER_BOUND)
)

n_excluded_low  = (df_data_ontario["salary_avg_annual"] < LOWER_BOUND).sum()
n_excluded_high = (df_data_ontario["salary_avg_annual"] > UPPER_BOUND).sum()

# original 1,113 rows remain in the DataFrame for audit, but downstream
# analyses can filter on the flag.
df_data_ontario["_salary_in_plausible_range"] = mask_in_range

print("Safety filter results (annualized salary outside [$25K, $500K]):")
print(f"  Excluded (too low):    {n_excluded_low} jobs")
print(f"  Excluded (too high):   {n_excluded_high} jobs")
print(f"  Total excluded:        {(~mask_in_range).sum()} jobs")
print(f"  Remaining in plausible range: {mask_in_range.sum()} jobs ({100*mask_in_range.sum()/len(df_data_ontario):.1f}%)")
print()


# 4 - Final sanity check on the cleaned distribution

df_clean = df_data_ontario[df_data_ontario["_salary_in_plausible_range"]]

print("Final cleaned salary distribution (after corrections + filter):")
print(df_clean["salary_avg_annual"].describe().round(0))

Safety filter results (annualized salary outside [$25K, $500K]):
  Excluded (too low):    1 jobs
  Excluded (too high):   1 jobs
  Total excluded:        2 jobs
  Remaining in plausible range: 1111 jobs (99.8%)

Final cleaned salary distribution (after corrections + filter):
count      1111.0
mean      92165.0
std       29693.0
min       26260.0
25%       72800.0
50%       96897.0
75%      103730.0
max      205442.0
Name: salary_avg_annual, dtype: float64


In [ ]:
# =============================================================================
# Save the cleaned analytical dataset
# =============================================================================
# We save ONLY the rows in plausible range (1,111 jobs) and ONLY the columns
# we'll need for downstream analysis.

In [34]:
# 1 - Filter to the row we keep

df_to_save = df_data_ontario[df_data_ontario["_salary_in_plausible_range"]].copy()

# 2 - Define logical group

# Logical groups:
#   1. Identifiers      -- to trace back to raw source
#   2. Role             -- NOC + cleaned title
#   3. Location         -- province / city / region
#   4. Compensation     -- raw + corrected + annualized
#   5. Profile fields   -- education, experience (for future seniority analysis)
#   6. Source tracking  -- which monthly file this came from

columns_to_save = [
    # 1. Identifiers
    "WIC Job Location Snapshot ID",
    "First Posting Date",
    # 2. Role
    "_noc21_clean",
    "Job Title",
    "NOC21 Code Name",                # human-readable label from the source
    # 3. Location
    "Province/Territory",
    "City",
    "Economic  Region",               # /!\ TWO spaces -- gov data typo
    # 4. Compensation
    "Salary Minimum",
    "Salary Maximum",
    "Salary Per",
    "_salary_per_corrected",
    "salary_min_annual",
    "salary_max_annual",
    "salary_avg_annual",
    # 5. Profile fields
    "Education LOS",
    "Experience Level",
    "Hours Minimum",                  # weekly hours range, low end
    "Hours Maximum",                  # weekly hours range, high end
    "Hours Per",                      # unit for Hours Min/Max (week, day, etc)
    # 6. Source tracking
    "_source_month",
]


#Sanity check
missing_cols = [c for c in columns_to_save if c not in df_to_save.columns]
if missing_cols:
    raise ValueError(f"Columns missing from df_to_save: {missing_cols}")

df_to_save = df_to_save[columns_to_save]

# 3 -  Write to data/processed/
output_path = Path ("../data/processed/jobbank_ontario_data_clean.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

df_to_save.to_csv(output_path, index=False, encoding="utf-8-sig")

# 4 - Confirm the save and report
size_kb = output_path.stat().st_size / 1024

print(f"Saved: {output_path.resolve()}")
print(f"  Rows:    {len(df_to_save):,}")
print(f"  Columns: {len(df_to_save.columns)}")
print(f"  Size:    {size_kb:,.1f} KB")
print()
print("Column list saved:")
for i, col in enumerate(df_to_save.columns, 1):
    print(f"  {i:>2}. {col}")


Saved: C:\Users\rebec\Documents\GitHub\toronto-data-job-market-analysis\data\processed\jobbank_ontario_data_clean.csv
  Rows:    1,111
  Columns: 21
  Size:    208.8 KB

Column list saved:
   1. WIC Job Location Snapshot ID
   2. First Posting Date
   3. _noc21_clean
   4. Job Title
   5. NOC21 Code Name
   6. Province/Territory
   7. City
   8. Economic  Region
   9. Salary Minimum
  10. Salary Maximum
  11. Salary Per
  12. _salary_per_corrected
  13. salary_min_annual
  14. salary_max_annual
  15. salary_avg_annual
  16. Education LOS
  17. Experience Level
  18. Hours Minimum
  19. Hours Maximum
  20. Hours Per
  21. _source_month


In [33]:
# Diagnostic

print(f"Total columns: {len(df_data_ontario.columns)}")
print()
for c in df_data_ontario.columns:
    print(f"  - {c!r}")

Total columns: 72

  - 'WIC Job Location Snapshot ID'
  - 'Job Title'
  - 'Original Job Title'
  - 'NOC 2016 Code'
  - 'NOC 2016 Code Name'
  - 'NOC21 Code'
  - 'NOC21 Code Name'
  - 'External Indicator'
  - 'First Posting Date'
  - 'Vacancy Count'
  - 'Official Language'
  - 'Education LOS'
  - 'Experience Level'
  - 'Government Type'
  - 'Placement Agency'
  - 'NAICS'
  - 'Province/Territory'
  - 'City'
  - 'Work Location Postal Code'
  - 'Economic  Region'
  - 'Various Location'
  - 'Employment Type'
  - 'Employment Term'
  - 'Employment Term Start Date'
  - 'Employment Term End Date'
  - 'Employment Term Oncall'
  - 'Employment Term Overtime'
  - 'Employment Term Day'
  - 'Employment Term Evening'
  - 'Employment Term Shift'
  - 'Employment Term Weekend'
  - 'Employment Term Night'
  - 'Employment Term Telework'
  - 'Employment Term Early'
  - 'Employment Term Flex'
  - 'Employment Term Morning'
  - 'Employment Term TBD'
  - 'Salary Condition Detail'
  - 'Salary Per'
  - 'Salary Mi